# ✂️ EWIT – Xử lý video theo Bảng điều khiển
Mọi thông tin (loại video, người chia sẻ, đoạn cắt, chế độ) lấy từ Google Sheet **Bảng điều khiển video – IT Team**.
Ở đây bạn chỉ cần nhập **Mã video** rồi bấm ▶ lần lượt từng ô.

## Bước 1 – Đăng nhập Google và tải tool
Cửa sổ hỏi quyền hiện ra → chọn tài khoản → **Cho phép** (để đọc/ghi Sheet và Drive).

In [ ]:
from google.colab import auth
auth.authenticate_user()
import subprocess, sys, os, json
subprocess.run('rm -rf /content/VideoApps && git clone -q --depth 1 -b claude/blissful-thompson-ey00yo '
               'https://github.com/tnho113/VideoApps /content/VideoApps', shell=True, check=True)
TOOL = '/content/VideoApps/tools/zoom_cut'
sys.path.insert(0, TOOL)
from sheet_job import Job, build_recipe, speaker_line, fmt
from transcript_compact import read_vtt
import cut_video, make_thumbnail
print('✅ Sẵn sàng')

## Bước 2 – Chọn video ✏️
`CHAY_THU = True`: chỉ xử lý 10 phút đầu, không đổi Trạng thái trên Sheet.

In [ ]:
MA_VIDEO = 'IT_T001'   # ✏️ Mã video trong tab Video
CHAY_THU = True        # ✏️ True = 10 phút đầu; False = cả video

job = Job(MA_VIDEO)
v = job.video
recipe = build_recipe(v, job.vtype, job.cuts)
print(f"🎬 {MA_VIDEO} | {v['Loại video']} | {v['Buổi số']} | {v['Tiêu đề']}")
print(f"👤 {speaker_line(v) or '(chưa chọn người)'}")
print(f"⚙️ Chế độ: {v['Chế độ']} | Trạng thái: {v['Trạng thái']} | Đoạn sẽ cắt (đã tick): {len(recipe['cat'])}")
if not recipe['cat']:
    print('⚠️ Chưa có đoạn cắt nào được tick trong tab Chi tiết cắt – video sẽ giữ nguyên.')
if not CHAY_THU and v['Chế độ'] == 'Chờ duyệt' and v['Trạng thái'] != 'Đã duyệt':
    raise SystemExit('⏸ Video ở chế độ Chờ duyệt: kiểm tra tab Chi tiết cắt, rồi đổi Trạng thái thành "Đã duyệt" và chạy lại ô này.')

## Bước 3 – Tải video và xem trước các đoạn sẽ cắt
Video 500 MB tải khoảng 1–3 phút. Đọc kỹ báo cáo trước khi sang Bước 4.

In [ ]:
mp4, vtt = job.find_inputs()
VIDEO = f"/content/{mp4['id']}.mp4"
if not os.path.exists(VIDEO):
    job.download(mp4['id'], VIDEO)
cues = read_vtt(job.download(vtt['id'], f"/content/{vtt['id']}.vtt")) if vtt else None
total = cut_video.duration(VIDEO)
until = 600 if CHAY_THU else None
cuts, keeps = cut_video.plan(recipe, total, until=until)
cut_video.report(cuts, keeps, min(total, until) if until else total, cues)

## Bước 4 – Cắt video và lưu vào Drive
Bản thử: vài phút. Cả video 150 phút: có thể 1–2 giờ, **giữ tab mở**. Kết quả nằm trong thư mục `ket_qua` của buổi và link được ghi vào Sheet.

In [ ]:
TEN = f'{MA_VIDEO}_thu10p' if CHAY_THU else MA_VIDEO
if not CHAY_THU: job.set(**{'Trạng thái': 'Đang cắt'})
print('⏳ Đang cắt…')
cut_video.cut(VIDEO, keeps, f'/content/{TEN}.mp4')
links = {'Video đã cắt': job.upload(f'/content/{TEN}.mp4')}
if cues:
    cut_video.write_srt(cues, keeps, f'/content/{TEN}.srt'); job.upload(f'/content/{TEN}.srt')
job.set(**{'Thời lượng gốc': fmt(total), 'Sau khi cắt': fmt(sum(b - a for a, b in keeps)), **links})
print('✅ Đã lưu video và ghi link vào Sheet:', links['Video đã cắt'])

## Bước 5 – Tạo thumbnail
Tự chọn **mẫu** (cột Mẫu thumbnail của video, hoặc mặc định theo Loại video), **kiểu ảnh** (Trong / Phủ ngoài vòng tròn) và **ảnh người** trong tab Người & Nhóm. Thiếu ảnh → không ảnh hoặc chibi; thiếu ảnh tách nền → tạm dùng kiểu Trong vòng tròn.

In [ ]:
note = job.thumbnail(f'/content/{TEN}.jpg')
job.set(Thumbnail=job.upload(f'/content/{TEN}.jpg'))
job.set(**{'Ghi chú': (v['Ghi chú'] + ' | ' + note).strip(' |') if CHAY_THU else note})
if not CHAY_THU: job.set(**{'Trạng thái': 'Xong'})
print('ℹ️', note)
from IPython.display import Image, display; display(Image(f'/content/{TEN}.jpg', width=640))

## Bước 6 – Đăng YouTube (làm sau khi bản cắt đã ổn)
Cần `client_secret.json` trong thư mục của buổi – xem `tools/zoom_cut/README.md`.

In [ ]:
DANG_YOUTUBE = False   # ✏️ đổi thành True khi đã sẵn sàng
if DANG_YOUTUBE:
    from upload_youtube import get_credentials, upload
    sec = job._list(f"'{job.folder}' in parents and name='client_secret.json' and trashed=false")
    job.download(sec[0]['id'], '/content/client_secret.json')
    creds = get_credentials('/content/client_secret.json', '/content/youtube_token.json')
    vid = upload(f'/content/{TEN}.mp4', v['Tiêu đề'], recipe['mo_ta'], recipe['the_tag'],
                 f'/content/{TEN}.jpg', 'private', creds, f'/content/{TEN}.srt' if cues else None)
    job.set(YouTube=f'https://youtu.be/{vid}')